# Working with files

**Question:** How do an animal's average behavioral scores differ before and after training on each day?

An animal completed 40 test trials before training (*initial*) and 40 after training (*final*) on each of six days. Each phase has its own file. A higher response is a higher behavioral score.

We will build one workflow: **inspect → discover → load → analyze → save → reload and plot**.

By the end of this lecture, you will be able to find data files, select the right column, keep each result associated with its day and phase, and generate output paths automatically.

## 1. Find your bearings

Download the notebook and the separate [**4.01_files_data.zip**](4.01_files_data.zip) archive into a working folder of your choice.

1. Extract the ZIP using your file manager. Browsing inside a ZIP is not the same as extracting it.
2. Find the extracted `behavioral_data` folder and place it beside this notebook. If extraction created an extra enclosing folder, move `behavioral_data` out of it.
3. Open `behavioral_data/20230914` in your file manager. You should see two CSV files and `broken.msg`.
4. Start the notebook from your working folder and run the directory check below. `Data folder found` should be `True`.

The ZIP stores the files for transport; the analysis reads the extracted files. Keep the ZIP after extraction.

After extraction, your working folder should contain:

```text
working_folder/
├── 4.01_files_batch_analysis.ipynb
├── 4.01_files_data.zip
└── behavioral_data/
```

Inside `behavioral_data`:

```text
behavioral_data/
├── 20230912/
│   ├── initial_behavioral_scores.csv
│   └── final_behavioral_scores.csv
├── 20230913/
│   ├── initial_behavioral_scores.csv
│   └── final_behavioral_scores.csv
├── 20230914/
│   ├── initial_behavioral_scores.csv
│   ├── final_behavioral_scores.csv
│   └── broken.msg
└── ... three more dated folders, through 20230917
```

A bit of nomenclature: Take this path `experiments/experiment_1/final_behavioral_scores.txt`
- `final_behavioral_scores.txt` is the file name, `final_behavioral_scores` is called the file stem, `.txt` is called the suffix or extension
- `experiments` and `experiment_1` are directories or folders. `experiments` is the parent directory of `experiment_1`. `experiment_1` is  a subdirectory of `experiments`
- `/` is the path separator. On windows, the path separator is '\'.

We can now list all data files, ending in `.txt` in the `experiment_*` subfolders:

Dates use `YYYYMMDD`. Alphabetical sorting therefore also gives chronological order.

A **relative path** starts at Python's current working directory. That directory is not necessarily the notebook's directory. Check it below if a file cannot be found.

In [ ]:
import os
from glob import glob

import numpy as np
import matplotlib.pyplot as plt

print("Working directory:", os.getcwd())
print("Data folder found:", os.path.isdir("behavioral_data"))

## 2. Inspect and load one file

Open `behavioral_data/20230914/final_behavioral_scores.csv` in a text editor. Its first line is `Trial,Response`, followed by 40 rows of numbers.

The comma is the **delimiter**, separating the two columns. The header describes the columns; it is not numerical data. A `.csv` or `.txt` extension is a naming convention: always inspect the contents.

In [ ]:
example_file = os.path.join("behavioral_data", "20230914", "final_behavioral_scores.csv")
example_data = np.loadtxt(example_file, delimiter=",", skiprows=1)
print("Shape:", example_data.shape)
print(example_data[:5])

`skiprows=1` skips the header. The array has 40 rows and two columns.

**Predict:** Which expression selects all responses: `example_data[1, :]` or `example_data[:, 1]`?

**Mini exercise:** Calculate the average response for the data in the loaded file. Save it in the variable `mean_response`.

In [ ]:
# your solution here
mean_response = 0

This is the mean for one phase on one day. Next, let's find the other files without typing every filename.

## 3. Discover files with patterns

`glob` returns matching paths. `*` matches zero or more characters within one path component; `?` matches one character. It does not guarantee a particular order, so use `sorted` when order matters.

In [ ]:
print("Everything in one day:")
print(sorted(glob(os.path.join("behavioral_data", "20230914", "*"))))

Check the files listed above. Do they all look like valid data files?

**Mini exercise:** Modify the `glob` statement to only list `.csv` files in that day.

In [ ]:
# your soliution here

**Mini exercise:** Thus far, we list files for a single day. Let's list all csv files from all days:

In [ ]:
# your solution here

There are **12** data files: six days × two phases. The pattern excludes `broken.msg`. Notice that alphabetically `final` comes before `initial`; file order does not tell us which phase belongs in which result column.

An empty list usually means the folder or pattern is wrong. Check the working directory and inspect a few matches before starting an analysis.

## 4. Take paths apart and build result paths

For `behavioral_data/20230914/final_behavioral_scores.csv`:

- `behavioral_data` is the data root; `20230914` is the day folder.
- `final_behavioral_scores.csv` is the filename.
- `final_behavioral_scores` is the stem; `.csv` is the extension.

We use `os.path` throughout this lecture. `pathlib` is another standard-library option; one consistent approach is enough here.

In [ ]:
folder, filename = os.path.split(example_file)
data_root, day = os.path.split(folder)
stem, extension = os.path.splitext(filename)

print("Data root:", data_root)
print("Day:", day)
print("Stem:", stem)
print("Extension:", extension)

**Goal:** Save this file's mean in `behavioral_results/20230914/final_behavioral_scores.txt`.

Keep the day and stem, change the root and extension. `os.path.join` uses the appropriate separator on your computer.

In [ ]:
result_folder = os.path.join("behavioral_results", day)
result_file = os.path.join(result_folder, stem + ".txt")
print(result_file)

A path is just a string; building one does not create a folder or file. We therefore need to create the folder before saving. `exist_ok=True` allows rerunning the cell when the folder already exists.

`np.savetxt` accepts a one- or two-dimensional array (or list), so put our single mean in a list. By default it prefixes header text with `#`; `np.loadtxt` ignores those comment lines.

**Rerunning save cells replaces files at the same output paths.** All outputs here go into `behavioral_results`, separate from the input data.

In [ ]:
os.makedirs(result_folder, exist_ok=True)
np.savetxt(result_file, [mean_response], header="Mean_response", fmt="%.6f")
print("Saved:", result_file)
print("Reloaded mean:", np.loadtxt(result_file))

## 5. First attempt: parallel lists

We now want to analyze all data and collect the results using python.

Let's list all data files. For this complete dataset, sorting both file lists puts them in matching date order. `zip` lets us visit corresponding items together. It stops at the end of the shorter list, so it does not warn us about missing files.

In [ ]:
initial_files = sorted(glob(os.path.join("behavioral_data", "*", "initial_behavioral_scores.csv")))
final_files = sorted(glob(os.path.join("behavioral_data", "*", "final_behavioral_scores.csv")))
print(initial_files)
print(final_files)

Now design the workflow:
1. for each day, we have to load the initlal and final scores
2. compute initial and final means 
3. collect the initial and final means in a data structure.

**Mini exercise:** Implement step 1 (load all files using a for loop). Then add step 2 (calculation of means) to the for loop.

In [ ]:
# your solution here

For step 3, we need to think about how to organize the results. A straightforward approach for representing the different results is to keep separate lists of days, initial means, and final means. We call them **parallel lists** because position `i` must refer to the same day in every list.

**Mini exercise:** Append the results as well as the day (extract it from the file path using `os.path.split`) to the respective lists.

In [ ]:
list_days = []
initial_means = []
final_means = []

# your solution here

## 6. A better representation: a dictionary of daily results

An alternative is to use dictionaries with one key per day to organize the results. That way, the association between days and results is explicit. 

Use the date as a dictionary key. Store the two means under the names `initial` and `final`:

```python
scores["20230912"] = {"initial": 0.087, "final": 0.262}
```

Then `scores["20230912"]["final"]` says exactly which value we want. To build this safely, discover the day folders once, then construct both input filenames inside **the same folder**. We do not pair independent file lists by position.

In [ ]:
day_folders = sorted(glob(os.path.join("behavioral_data", "*")))
scores = {}

for day_folder in day_folders:
    day = os.path.basename(day_folder)
    initial_file = os.path.join(day_folder, "initial_behavioral_scores.csv")
    final_file = os.path.join(day_folder, "final_behavioral_scores.csv")
    initial_data = np.loadtxt(initial_file, delimiter=",", skiprows=1)
    final_data = np.loadtxt(final_file, delimiter=",", skiprows=1)
    scores[day] = {
        "initial": np.mean(initial_data[:, 1]),
        "final": np.mean(final_data[:, 1]),
    }

print(scores["20230912"])
print("Final mean on September 12:", scores["20230912"]["final"])

| Parallel lists | Dictionary of daily records |
|---|---|
| Simple to create and pass to plotting functions | Names make each value's meaning explicit |
| Every list must be reordered and filtered together | Sorting keys retrieves complete daily records |
| Matching positions can silently refer to different days | Nested indexing and conversion to a table take an extra step |


## 7. Save a table with its day labels

A dictionary is useful during analysis. A rectangular table is convenient for sharing. Build each row from **one dictionary entry**: day, initial mean, final mean.

Dates remain strings in the dictionary. For this numeric text table only, we write each `YYYYMMDD` label as an integer.

In [ ]:
summary_rows = []
for day in sorted(scores):
    daily_scores = scores[day]
    summary_rows.append([int(day), daily_scores["initial"], daily_scores["final"]])
summary = np.array(summary_rows)

summary_file = os.path.join("behavioral_results", "scores.csv")
np.savetxt(
    summary_file, summary, delimiter=",", header="Day,Initial,Final",
    comments="", fmt=["%s", "%.6f", "%.6f"],
)
print("Saved rows:", summary.shape[0])

The first column preserves which day each row represents. `fmt` writes the date without decimals and the means to six decimal places. `comments=""` writes the header without `#`, so we skip it when reloading.

Reload the file using `np.loadtxt` to check what we saved before plotting it.

In [ ]:
loaded_summary = np.loadtxt(summary_file, delimiter=",", skiprows=1)
np.testing.assert_allclose(loaded_summary, summary, rtol=0, atol=0.0000005)
print(loaded_summary)

**Mini exercise:** Plot the results - days are on the x-axis, two lines with the initial and final scores across days.

In [ ]:
# your solition

Each point summarizes 40 trials in one phase on one day. In these data, the final mean is higher on every day. The plot describes the observed scores; by itself it does not establish a causal effect of training.

## 8. Save NumPy arrays: `.npy` and `.npz`

Text tables are easy to inspect and share. NumPy's binary formats preserve array shape and dtype without choosing a decimal text format.

Use `.npy` for one array. Here we save the two-column example table.

In [ ]:
array_file = os.path.join("behavioral_results", "example_data.npy")
np.save(array_file, example_data)
loaded_array = np.load(array_file)
print(loaded_array.shape)

Use `.npz` for several **named arrays**. Keep the dates and means together in one file. We are saving arrays, not the nested dictionary itself.

`np.load` returns a dictionary-like archive for `.npz`. Call `archive.close()` after retrieving the arrays.

In [ ]:
archive_file = os.path.join("behavioral_results", "scores.npz")
np.savez(archive_file, days=np.array(sorted(scores)), means=summary[:, 1:])

archive = np.load(archive_file)
print(archive)
print(archive["days"])
print(archive["means"])
archive.close()

## 9. Loading Excel data

Excel workbooks contain sheets, each of which can hold a table. We can load a sheet with pandas and convert its table to a NumPy array.

The [**4.01_files_data.zip**](4.01_files_data.zip) archive contains `format_examples`. Extract it beside `behavioral_data`, so both folders are beside this notebook. The Excel workbook contains response measurements for two mice.

First try loading the workbook with pandas; it normally selects an Excel reader automatically. The example is supplied as `.xlsx`. [pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html)

**Only if you get an error about a missing Excel reader:** install `openpyxl` in your notebook's Python environment:

```bash
conda install openpyxl
```

If you are using the browser version and get this error, run `%pip install openpyxl` in a new code cell, then rerun the loading cell.

In [ ]:
import pandas as pd

excel_file = os.path.join("format_examples", "mouse_data.xlsx")
workbook = pd.ExcelFile(excel_file)
print(workbook.sheet_names)
workbook.close()

Load the sheet named `Mouse1`. By default, pandas uses the first row as column names; no delimiter is needed.

The result is a **DataFrame**: a table with named columns, rather than a NumPy array.

In [ ]:
excel_data = pd.read_excel(excel_file, sheet_name="Mouse1")
print(type(excel_data))
print(excel_data.head())

Use `.to_numpy()` to continue with the NumPy operations we already know. This sheet contains only one column, `Responses`, so it is column `0`. Conversion keeps the values but leaves the column names behind.

In [ ]:
excel_array = excel_data.to_numpy()
print("Shape:", excel_array.shape)
print("Mean response:", np.mean(excel_array[:, 0]))

**Mini exercise:** Load the `Mouse2` sheet from the same workbook. Inspect its columns, convert it to a NumPy array, and calculate the mean response.

In [ ]:
# your solution here

## 10. Loading MATLAB data

A MATLAB `.mat` file can store several named variables. `scipy.io.loadmat` returns a **dictionary**, with variable names as keys and their values as arrays.

The example contains `initial` and `final`: the two-column trial/response tables for **20230914**, copied from our CSV files. Inspect the keys before choosing a variable. [SciPy documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.loadmat.html)

In [ ]:
from scipy.io import loadmat

matlab_file = os.path.join("format_examples", "behavioral_scores_20230914.mat")
matlab_data = loadmat(matlab_file)
print(matlab_data.keys())

Keys such as `__header__` and `__version__` describe the file. Our data variables are `initial` and `final`.

Select the final table by name, inspect its shape, and select the response column before calculating a mean. MATLAB arrays can retain two dimensions, even when they contain a single row or column, so check the shape rather than assuming it.

In [ ]:
matlab_final = matlab_data["final"]
print("Shape:", matlab_final.shape)
print(matlab_final[:5])
print("Mean final response:", np.mean(matlab_final[:, 1]))

**MATLAB format note:** `loadmat` supports v4 and v6–v7.2 MAT-files. MATLAB v7.3 uses HDF5 and needs another reader; `h5py` can access its datasets, but MATLAB structs and cell arrays may need additional decoding. This example uses the format supported by SciPy. [Format support](https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.loadmat.html)